In [1]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd().parent
raw_dir = base_dir / "data/raw"

dfs = {}

dtype_config = {
    "customers": {
        "customer_zip_code_prefix": "str"
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "str"
    },
    "sellers": {
        "seller_zip_code_prefix": "str"
    },
    "products": {
        "product_name_lenght": "Int64",
        "product_description_lenght": "Int64",
        "product_photos_qty": "Int64"
    }
}

datetime_config = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for file in raw_dir.glob("*.csv"):
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    dfs[name] = pd.read_csv(
        file,
        dtype=dtype_config.get(name, {})
    )
    if name in datetime_config:
        for col in datetime_config[name]:
            dfs[name][col] = pd.to_datetime(dfs[name][col])

orders = dfs["orders"]
customers = dfs["customers"]
geolocation = dfs["geolocation"]
sellers = dfs["sellers"]
products = dfs["products"]
category_name_translation = dfs["product_category_name_translation"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]

# Orders

Non ci sono righe duplicate.

Le seguenti colonne presentano valori null:

* *order_approved_at*
* *order_delivered_carrier_date*
* *order_delivered_customer_date*

Sono state rilevate le seguenti incoerenze temporali:

* 166 ordini con *order_purchase_timestamp* > *order_delivered_carrier_date*
* 1359 ordini con *order_approved_at* > *order_delivered_carrier_date*
* 23 ordini con *order_delivered_carrier_date* > *order_delivered_customer_date*

In [2]:
orders.info(), orders.shape, orders.head(), orders.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


(None,
 (99441, 8),
                            order_id                       customer_id  \
 0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
 1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
 2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
 3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
 4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
 
   order_status order_purchase_timestamp   order_approved_at  \
 0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
 1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
 2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
 3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
 4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   
 
   order_delivered_carrier_date order_delivered_customer_date  \
 0          2017-10-04 19:55:00           2017-10-10 21:25:13   
 1          2018-0

In [3]:
(
    orders["order_purchase_timestamp"] >
    orders["order_approved_at"]
).sum()

np.int64(0)

In [4]:
(
    orders["order_purchase_timestamp"] >
    orders["order_delivered_carrier_date"]
).sum()

np.int64(166)

In [5]:
(
    orders["order_approved_at"] >
    orders["order_delivered_carrier_date"]
).sum()

np.int64(1359)

In [6]:
(
    orders["order_purchase_timestamp"] >
    orders["order_delivered_customer_date"]
).sum()

np.int64(0)

In [7]:
(
    orders["order_delivered_carrier_date"] >
    orders["order_delivered_customer_date"]
).sum()

np.int64(23)

# Customers

Non ci sono righe duplicate o valori null.

In [8]:
customers.info(), customers.shape, customers.head(), customers.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


(None,
 (99441, 5),
                         customer_id                customer_unique_id  \
 0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 
   customer_zip_code_prefix          customer_city customer_state  
 0                    14409                 franca             SP  
 1                    09790  sao bernardo do campo             SP  
 2                    01151              sao paulo             SP  
 3                    08775        mogi das cruzes             SP  
 4                    13056               campinas             SP  ,
 customer_id                 99441
 customer_unique_id          96096
 customer_zip_code_prefix    14994
 customer_city      

# Geolocation

Ci sono 261.831 righe duplicate.

Non ci sono valori null.

Le seguenti colonne presentano valori negativi:

* *geolocation_lat*
* *geolocation_lng*

In [9]:
geolocation.info(), geolocation.shape, geolocation.head(), geolocation.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(3)
memory usage: 38.2 MB


(None,
 (1000163, 5),
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
 0                       01037       -23.545621       -46.639292   
 1                       01046       -23.546081       -46.644820   
 2                       01046       -23.546129       -46.642951   
 3                       01041       -23.544392       -46.639499   
 4                       01035       -23.541578       -46.641607   
 
   geolocation_city geolocation_state  
 0        sao paulo                SP  
 1        sao paulo                SP  
 2        sao paulo                SP  
 3        sao paulo                SP  
 4        sao paulo                SP  ,
 geolocation_zip_code_prefix     19015
 geolocation_lat                717360
 geolocation_lng                717613
 geolocation_city                 8011
 geolocation_state                  27
 dtype: int64)

In [10]:
geolocation.duplicated().sum()

np.int64(261831)

# Sellers

Non ci sono righe duplicate o valori null.

In [11]:
sellers.info(), sellers.shape, sellers.head(), sellers.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 96.8 KB


(None,
 (3095, 4),
                           seller_id seller_zip_code_prefix        seller_city  \
 0  3442f8959a84dea7ee197c632cb2df15                  13023           campinas   
 1  d1b65fc7debc3361ea86b5f14c68d2e2                  13844         mogi guacu   
 2  ce3ad9de960102d0677a81f5d0bb7b2d                  20031     rio de janeiro   
 3  c0f3eea2e14555b6faeea3dd58c1b1c3                  04195          sao paulo   
 4  51a04a8a6bdcb23deccc82b0b80742cf                  12914  braganca paulista   
 
   seller_state  
 0           SP  
 1           SP  
 2           RJ  
 3           SP  
 4           SP  ,
 seller_id                 3095
 seller_zip_code_prefix    2246
 seller_city                611
 seller_state                23
 dtype: int64)

# Products

Non ci sono righe duplicate.

Ogni colonna, ad eccezione di *product_id*, presenta valori null. In particolare, 610 prodotti presentano, contemporaneamenente, valori null nelle seguenti colonne:

* *product_category_name*
* *product_name_lenght*
* *product_description_lenght*
* *product_photos_qty*

Soltanto 2 ordini, invece, presentano, contemporaneamente, valori null nelle seguenti colonne:

* *product_weight_g*
* *product_length_cm*
* *product_height_cm*
* *product_width_cm*

Le seguenti colonne presentano un errore ortografico (*lenght* invece di *length*):

* *product_name_lenght*
* *product_description_lenght*

In [12]:
products.info(), products.shape, products.head(), products.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  Int64  
 3   product_description_lenght  32341 non-null  Int64  
 4   product_photos_qty          32341 non-null  Int64  
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: Int64(3), float64(4), str(2)
memory usage: 2.4 MB


(None,
 (32951, 9),
                          product_id  product_category_name  \
 0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
 1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
 2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
 3  cef67bcfe19066a932b7673e239eb23d                  bebes   
 4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   
 
    product_name_lenght  product_description_lenght  product_photos_qty  \
 0                   40                         287                   1   
 1                   44                         276                   1   
 2                   46                         250                   1   
 3                   27                         261                   1   
 4                   37                         402                   4   
 
    product_weight_g  product_length_cm  product_height_cm  product_width_cm  
 0             225.0               16.0               10.0           

In [13]:
products.query("product_category_name.isnull()")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,<NA>,<NA>,<NA>,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,<NA>,<NA>,<NA>,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,<NA>,<NA>,<NA>,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,<NA>,<NA>,<NA>,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,<NA>,<NA>,<NA>,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,<NA>,<NA>,<NA>,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,<NA>,<NA>,<NA>,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,<NA>,<NA>,<NA>,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,<NA>,<NA>,<NA>,1300.0,45.0,16.0,45.0


In [14]:
products.query(
    "product_category_name.isna() and product_name_lenght.isna() \
    and product_description_lenght.isna() and product_photos_qty.isna()"
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,<NA>,<NA>,<NA>,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,<NA>,<NA>,<NA>,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,<NA>,<NA>,<NA>,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,<NA>,<NA>,<NA>,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,<NA>,<NA>,<NA>,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,<NA>,<NA>,<NA>,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,<NA>,<NA>,<NA>,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,<NA>,<NA>,<NA>,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,<NA>,<NA>,<NA>,1300.0,45.0,16.0,45.0


In [15]:
products.query(
    "product_weight_g.isna() and product_length_cm.isna() \
    and product_height_cm.isna() and product_width_cm.isna()"
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN


# Category Name Translation

Non ci sono righe duplicate o valori null.

Ci sono 2 *product_category_name* in meno rispetto a quelli presenti nella tabella *products*.

In [16]:
category_name_translation.info(), category_name_translation.shape,\
category_name_translation.head(), category_name_translation.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


(None,
 (71, 2),
     product_category_name product_category_name_english
 0            beleza_saude                 health_beauty
 1  informatica_acessorios         computers_accessories
 2              automotivo                          auto
 3         cama_mesa_banho                bed_bath_table
 4        moveis_decoracao               furniture_decor,
 product_category_name            71
 product_category_name_english    71
 dtype: int64)

# Order Items

Non ci sono righe duplicate e valori null.

In [17]:
order_items.info(), order_items.shape, order_items.head(), order_items.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


(None,
 (112650, 7),
                            order_id  order_item_id  \
 0  00010242fe8c5a6d1ba2dd792cb16214              1   
 1  00018f77f2f0320c557190d7a144bdd3              1   
 2  000229ec398224ef6ca0657da4fc703e              1   
 3  00024acbcdf0a6daa1e931b038114c75              1   
 4  00042b26cf59d7ce69dfabb4e55b4fd9              1   
 
                          product_id                         seller_id  \
 0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
 1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
 2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
 3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
 4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   
 
   shipping_limit_date   price  freight_value  
 0 2017-09-19 09:45:35   58.90          13.29  
 1 2017-05-03 11:05:13  239.90          19.93  
 2 2018-01-18 14:48:30  199.00          17.87  
 3 2018-0

In [18]:
order_items.duplicated().sum()

np.int64(0)

# Order Payments

Non ci sono righe duplicate e valori null.

Ci sono 80 *order_id* con una sequenza errata di *payment_sequential* che vanno gestiti.

La colonna *payment_type* presenta dei valori *not_defined* che vanno gestiti.

La colonna *payment_installments* presenta valori pari a 0 che vanno gestiti in quanto il numero di rate minimo accettato è 1, ossia il pagamento in un'unica soluzione.

In [19]:
order_payments.info(), order_payments.shape,\
order_payments.head(), order_payments.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


(None,
 (103886, 5),
                            order_id  payment_sequential payment_type  \
 0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
 1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
 2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
 3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
 4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   
 
    payment_installments  payment_value  
 0                     8          99.33  
 1                     1          24.39  
 2                     1          65.71  
 3                     8         107.78  
 4                     2         128.45  ,
 order_id                99440
 payment_sequential         29
 payment_type                5
 payment_installments       24
 payment_value           29077
 dtype: int64)

In [20]:
order_payments.duplicated().sum()

np.int64(0)

In [21]:
(
    order_payments
        .groupby("order_id")["payment_sequential"]
        .apply(
            lambda x: sorted(x.tolist()) != list(range(1, x.max() + 1))
        )
).sum()

np.int64(80)

In [22]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [23]:
order_payments.query("payment_installments < 1")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


# Order Reviews

Non ci sono righe duplicate.

Le seguenti colonne contengono valori null:

* *review_comment_title*
* *review_comment_message*

In [24]:
order_reviews.info(), order_reviews.shape, order_reviews.head(), order_reviews.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


(None,
 (99224, 7),
                           review_id                          order_id  \
 0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
 1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
 2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
 3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
 4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   
 
    review_score review_comment_title  \
 0             4                  NaN   
 1             5                  NaN   
 2             5                  NaN   
 3             5                  NaN   
 4             5                  NaN   
 
                               review_comment_message review_creation_date  \
 0                                                NaN           2018-01-18   
 1                                                NaN           2018-03-10   
 2                                                Na

In [25]:
order_reviews.duplicated().sum()

np.int64(0)